In [24]:
import numpy as np
import pandas as pd
import math
import optuna
from tqdm import tqdm
import gc
import psutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn import TransformerDecoder, TransformerDecoderLayer, TransformerEncoder, TransformerEncoderLayer
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import matthews_corrcoef
from transformers import AutoTokenizer, AutoModel
from functools import lru_cache
from torch.amp import GradScaler, autocast

#Clear the GPU memory cache
torch.cuda.empty_cache()
pd.options.mode.chained_assignment = None  # Suppress the warning globally

import wandb
import os
import json
import pickle

from torchcrf import CRF
import ast
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp


#Clear the GPU memory cache
torch.cuda.empty_cache()

#Configure CUDA memory allocations (helps manage fragmentation in the GPU memory)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

In [25]:
#Change input directory path for ERDA
input_data_dir_path = "/tmp/nrt204/FragmentPredictor2/data/processed_data/model_data/shared_crf/model_with_errors" #TEST ON SCARB CLUSTER
input_data_dir_path = None #Local tests

if input_data_dir_path == None: 
    input_data_dir_path = "../../../data/processed_data/model_data/shared_crf/model_with_errors"

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device_type = device.type  # "cuda", "mps", or "cpu"

if device_type == "mps":
    num_workers_cpu = 0 #Adjust if not using mps
    pin_memory = False #Turn true if using CUDA
elif device_type == "cuda":
    num_workers_cpu = 4
    pin_memory = True

print("Device type:", device_type, flush = True)

#Model choice
esm2_model = "facebook/esm2_t6_8M_UR50D"
#esm2_model = "facebook/esm2_t33_650M_UR50D"
esm2_model_abbr = esm2_model.split("/")[-1].split("_UR")[0]
project_name = esm2_model_abbr

#Make sure dir to store model exists
os.makedirs(f"../../../data/processed_data/model_data/shared_crf/model_with_errors/models/", exist_ok=True)

#dir in wandb to place run
wandb_project_name = "optuna_shared_crf_errors_codon_encoding" #MODIFY!

max_len = 100 

Device type: mps


In [27]:
def set_seed(seed):
    """
    Set seed for reproducibility

    Args:
        seed (int): seed value to set
    """
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def clear_memory():
    #Memory clean up function
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

In [28]:
genetic_code = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '<unk>', 'TAG': '<unk>',
    'TGT': 'C', 'TGC': 'C', 'TGA': '<unk>', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate_nucleotide_to_amino_acid(sequence):
    """
    The function takes a nucleotide sequence and translates it into the corresponding amino acid sequence. 
    
    Args:
        sequence (str): A nucleotide sequence.
        
    Returns:
        (str): A string representing the amino acid sequence translated from the 
               nucleotide input. Each codon is mapped to its corresponding amino acid,
               with stop codons represented by "<unk>".
    """

    sequence = str(sequence)
    
    # More efficient length adjustment
    remainder = len(sequence) % 3
    if remainder:
        sequence = sequence[:-remainder]
    
    # Direct list comprehension is fastest
    return ''.join(genetic_code.get(sequence[i:i+3], "<mask>") 
                   for i in range(0, len(sequence), 3))

In [29]:
def one_hot_encode(sequence):
    """
    One-hot encode nucleotide sequences in a matrix format of 4 rows (A, C, G, T)
    and len(sequence) columns.

    Args:
        sequence (str): a nucleotide sequence. 

    Returns: 
        encoding (tensor): one-hot encoded sequence. 
    """

    #Define the mapping of nucleotides to indices
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    #Create an empty one-hot encoding tensor
    encoding = torch.zeros(4, len(sequence))
    
    #Convert the sequence to a tensor of indices for efficient indexing
    indices = torch.tensor(
        [mapping[char] for char in sequence if char in mapping], dtype=torch.long)
    
    #Use advanced indexing to set the appropriate positions to 1
    positions = torch.arange(len(sequence))[[char in mapping for char in sequence]]
    encoding[indices, positions] = 1
    #For 'N', we do nothing, so the corresponding column remains all zeros

    return torch.tensor(encoding)


In [30]:
def process_nt_sequences_to_codons(nt_sequences, max_aa_len):
    """
    Convert nucleotide sequences from (4, 300) to (12, 100) format
    by grouping every 3 nucleotides (1 codon) together.
    
    Args:
        nt_sequences: List of tensors with shape (4, 300)
    
    Returns:
        List of tensors with shape (12, 100)
    """
    processed_sequences = []
    
    for seq in nt_sequences:
        # seq has shape (4, 300)
        # Reshape to (4, 100, 3) to group every 3 nucleotides
        seq_reshaped = seq.view(4, max_aa_len, 3)
        
        # Transpose to (100, 4, 3) then reshape to (100, 12)
        seq_transposed = seq_reshaped.transpose(0, 1)  # (100, 4, 3)
        seq_formatted = seq_transposed.reshape(max_aa_len, 12)  # (100, 12)
         
        processed_sequences.append(seq_formatted)
    
    return processed_sequences

In [31]:
class SeqDataset(torch.utils.data.Dataset):
    """
    Optimized dataset class with reduced memory overhead.
    """
    def __init__(self, nt_encodings_rf0, labels_rf0, 
                 nt_encodings_rf1, labels_rf1, 
                 nt_encodings_rf2, labels_rf2, 
                 label_encodings,
                 seq_desc):
        
        self.nt_encodings_rf0 = nt_encodings_rf0
        self.labels_rf0 = labels_rf0  

        self.nt_encodings_rf1 = nt_encodings_rf1
        self.labels_rf1 = labels_rf1  

        self.nt_encodings_rf2 = nt_encodings_rf2
        self.labels_rf2 = labels_rf2  

        self.label_encodings = label_encodings
        self.seq_desc = seq_desc
    
    def __getitem__(self, idx):
        item = {
            'nt_encodings_rf0': torch.as_tensor(self.nt_encodings_rf0[idx], dtype=torch.float32),
            'labels_rf0': torch.from_numpy(self.labels_rf0[idx]) if isinstance(self.labels_rf0[idx], np.ndarray) else torch.tensor(self.labels_rf0[idx], dtype=torch.float32),

            'nt_encodings_rf1': torch.as_tensor(self.nt_encodings_rf1[idx], dtype=torch.float32),
            'labels_rf1': torch.from_numpy(self.labels_rf1[idx]) if isinstance(self.labels_rf1[idx], np.ndarray) else torch.tensor(self.labels_rf1[idx], dtype=torch.float32),

            'nt_encodings_rf2': torch.as_tensor(self.nt_encodings_rf2[idx], dtype=torch.float32),
            'labels_rf2': torch.from_numpy(self.labels_rf2[idx]) if isinstance(self.labels_rf2[idx], np.ndarray) else torch.tensor(self.labels_rf2[idx], dtype=torch.float32),

            'label_encodings': torch.from_numpy(self.label_encodings[idx]) if isinstance(self.label_encodings[idx], np.ndarray) else torch.tensor(self.label_encodings[idx], dtype=torch.float32),
            'seq_desc': self.seq_desc[idx]
        }
        return item
    
    def __len__(self):
        return len(self.label_encodings)


In [32]:
def encode_data(processed_samples_df, max_len):
    """ 
    Encode data samples to fit model input format. 

    Args:
        processed_samples_df (dataframe): Dataframe with input dataset.
    
    Returns:
        dataset (dict): nested dictionary with data formatted to fit model input.
        label_counts (dict): counts of each label in the dataset.
    """
    
    encodings_nt = {}
    labels = {}
    max_nt_len = max_len * 3

    #Label processing; shared mapped label sequence 
    if isinstance(processed_samples_df["label_encodings"].iloc[0], str):
        processed_samples_df["label_encodings"] = processed_samples_df["label_encodings"].apply(eval)

    label_arrays = [np.array(x, dtype=np.int8) for x in processed_samples_df["label_encodings"]]
    pad_positions = max_len
    padded_labels = np.full((len(label_arrays), pad_positions), -1, dtype=np.int8)

    for i, arr in enumerate(label_arrays):
        length = min(len(arr), pad_positions)
        padded_labels[i, :length] = arr[:length]
    
    #Proces data from each RF separately 
    for rf in ["rf0", "rf1", "rf2"]:
        
        #====Label processing====#
        if isinstance(processed_samples_df[f"{rf}_labels"].iloc[0], str):
            processed_samples_df[f"{rf}_labels"] = processed_samples_df[f"{rf}_labels"].apply(eval)
    
        #Convert to numpy arrays more efficiently
        label_arrays = [np.array(x, dtype=np.int8) for x in processed_samples_df[f"{rf}_labels"]]

        labels[rf] = label_arrays

        #====Nucleotide sequence processing====#
        # Pad the sequences
        processed_samples_df[f"{rf}_seq_nt"] = processed_samples_df[f"{rf}_seq_nt"].apply(
            lambda seq: seq + 'N' * (max_nt_len - len(seq)) if len(seq) < max_nt_len else seq)
        
        nt_sequences = [one_hot_encode(seq) for seq in processed_samples_df[f"{rf}_seq_nt"]] #[max_nt_len, 4, num_seqs]

        # Process nt_sequences to codon-based format (3*4=12, max_nt_len/3)
        nt_encodings_rf = process_nt_sequences_to_codons(nt_sequences, max_len)
        encodings_nt[rf] = nt_encodings_rf


    seq_descriptions = processed_samples_df["seq_desc"].tolist()

    dataset = SeqDataset(encodings_nt["rf0"], labels["rf0"],
                         encodings_nt["rf1"], labels["rf1"],
                         encodings_nt["rf2"], labels["rf2"],
                         padded_labels, seq_descriptions)
    
    return dataset, list(set(seq_descriptions))

In [33]:
def load_and_process_data(max_len):
    """
    Main function that loads and processes all data efficiently.
    """
    #Load data
    train_set = pd.read_csv(
        f"{input_data_dir_path}/datasets_model/train_small.csv.gz", #MODIFY
        index_col=None, 
        compression="gzip"
    )
    val_set = pd.read_csv(
        f"{input_data_dir_path}/datasets_model/val.csv.gz", #MODIFY
        index_col=None, 
        compression="gzip"
    )

    val_set = val_set.sample(frac=0.02, random_state=42)

    seq_type_desc_fracs = (val_set['seq_desc'].value_counts(normalize=True)).to_dict()
    
    print("Training data samples: ", train_set.shape[0])
    print("Validation data samples: ", val_set.shape[0])
    
    
    # Process training data
    train_data, sequence_types = encode_data(train_set, max_len)
    
    # Process validation data
    val_data, _ = encode_data(val_set, max_len)

    return train_data, val_data, sequence_types, seq_type_desc_fracs

# Model framework with CRF

In [34]:
def create_codon_padding_mask(frame_sequences):
    """
    Create padding mask for codon sequences.
    
    Args:
        frame_sequences: Tensor of shape (batch, 100, 12)
                        One-hot encoded codons
    
    Returns:
        mask: Boolean tensor (batch, 100)
              True = padding position (ignore in attention)
              False = real codon (use in attention)
    """
    # Sum across the 12 dimensions - padded positions will be all zeros
    mask = (frame_sequences.sum(dim=-1) == 0)
    return mask

In [35]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        # Pre-compute for efficiency, but can extend dynamically
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * 
                            -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        seq_len = x.size(1)
        
        if seq_len > self.pe.size(0):
            # Dynamically compute for longer sequences
            pe = self._compute_pe(seq_len, self.d_model, x.device)
            return x + pe[:seq_len, :]
        
        return x + self.pe[:seq_len, :]
    
    @staticmethod
    def _compute_pe(seq_len, d_model, device):
        pe = torch.zeros(seq_len, d_model, device=device)
        position = torch.arange(0, seq_len, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, device=device) * 
                            -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

In [36]:
class TransformerEncoderBlock(nn.Module):
    """
    Neural network module that applies a Transformer encoder block to encoded sequences and adds a linear layer to fit CRF input.
    
    Args:
        hidden_size (int): The dimensionality of the input and output features for the Transformer encoder.
        num_layers (int): Number of Transformer encoder layers to stack.
        n_attention_heads (int): Number of attention heads in each Transformer encoder layer.
        dropout_rate (float): Dropout rate applied after normalization and within the encoder layers.
        act_function (str or Callable): Activation function to use in the feedforward network of the encoder layers.
        num_labels (int): Number of output classes 
        
    Attributes:
        encoder (nn.TransformerEncoder): Stacked Transformer encoder layers.
        classifier (nn.Linear): Linear layer mapping the encoder output to class logits.
        norm (nn.LayerNorm): Layer normalization applied after the encoder.
        dropout (nn.Dropout): Dropout layer applied after normalization.
        layers (int): Number of encoder layers.
    """
    def __init__(self,
                 hidden_size,
                 num_layers,
                 n_attention_heads,
                 dropout_rate_encoder,
                 dropout_rate_final,
                 act_function,
                 num_labels):
        super().__init__()

        hidden_size_merged = hidden_size
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size_merged,
            nhead=n_attention_heads,
            dim_feedforward=4*hidden_size_merged,
            dropout=dropout_rate_encoder,
            activation=act_function
        )
        
        self.layers = num_layers
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.linear = nn.Linear(hidden_size_merged, num_labels)
        self.norm = nn.LayerNorm(hidden_size_merged)
        self.dropout_final = nn.Dropout(dropout_rate_final)


    def forward(self, encoded_seqs_nt, attention_mask):
        """
        Forward pass through the Transformer encoder block and linear classifier.
        
        Args:
            x (torch.Tensor): Input tensor [batch_size, seq_len, hidden_size]
            attention_mask (torch.Tensor): Attention mask [batch_size, seq_len]
            
        Returns:
            torch.Tensor of logits [batch_size, seq_len, num_labels]
        """

        # Original transformer processing
        codon_embeddings = encoded_seqs_nt.permute(1, 0, 2)  # [seq_len, batch, hidden + 3*4]
        attention_mask_transformer = ~attention_mask.bool()

        codon_embeddings = self.encoder(codon_embeddings, src_key_padding_mask=attention_mask_transformer)

        codon_embeddings = codon_embeddings.permute(1, 0, 2)  # [batch, seq_len, hidden]

        codon_embeddings = self.norm(codon_embeddings)
        codon_embeddings = self.dropout_final(codon_embeddings)

        logits = self.linear(codon_embeddings)  # [batch, seq_len, num_labels]

        return logits

In [37]:
class LinearChainCRF(nn.Module):
    """
    Neural network module that adds a CRF layer for structured prediction.
    Updated for dynamic RF combination labels.
    """
    def __init__(self, 
                 mapping_dict_to_class, 
                 crf_init_weight, 
                 num_class_labels=None):
        super().__init__()
        
        # Load the dynamic mapping
        self.label_to_rf = mapping_dict_to_class
        
        # Determine number of classes if not provided
        if num_class_labels is None:
            num_class_labels = len(self.label_to_rf)
        
        self.crf = CRF(num_tags=num_class_labels, batch_first=True)
        
        # RF transition rules
        self.legal_transitions = {
            0: {0, 2, 4},
            1: {1, 3, 5},
            2: {1, 5},
            3: {0, 2, 4},
            4: {1, 3, 5},
            5: {0, 2, 4}
        }
        
        self.constraint_mask = torch.ones_like(self.crf.transitions, dtype=torch.bool)
        self._create_constraint_mask()
        
        # Initialize transitions
        with torch.no_grad():
            self.crf.transitions[~self.constraint_mask] = -crf_init_weight  # forbidden → -2
            self.crf.transitions[self.constraint_mask] = crf_init_weight    # allowed → 1
    
    def _is_legal_transition(self, from_rf, to_rf):
        """Check if transition from one RF combination to another is legal"""
        from_rf0, from_rf1, from_rf2 = from_rf
        to_rf0, to_rf1, to_rf2 = to_rf
        
        # All three RFs must have legal transitions
        rf0_legal = to_rf0 in self.legal_transitions[from_rf0]
        rf1_legal = to_rf1 in self.legal_transitions[from_rf1]
        rf2_legal = to_rf2 in self.legal_transitions[from_rf2]
        
        return rf0_legal and rf1_legal and rf2_legal
    
    def _create_constraint_mask(self):
        """
        Create a mask indicating which transitions should be constrained.
        False = constrained (will be set to penalty), True = learnable
        """
        num_labels = len(self.label_to_rf)
        print(f"Creating constraint mask for {num_labels}-label CRF...")
        legal_count = 0
        illegal_count = 0
        
        # Check all possible transitions between the labels
        for from_label in range(num_labels):
            for to_label in range(num_labels):
                from_rf = self.label_to_rf[from_label]
                to_rf = self.label_to_rf[to_label]
                
                if self._is_legal_transition(from_rf, to_rf):
                    # Legal transition - keep as learnable (True)
                    self.constraint_mask[from_label, to_label] = True
                    legal_count += 1
                else:
                    # Illegal transition - mark as constrained (False)
                    self.constraint_mask[from_label, to_label] = False
                    illegal_count += 1
        
        print(f"Legal transitions: {legal_count}")
        print(f"Illegal transitions: {illegal_count}")
        print(f"Total transitions: {legal_count + illegal_count}")
        print(f"Percentage legal: {legal_count/(legal_count + illegal_count)*100:.1f}%")
    
    def forward(self, logits, attention_mask, labels=None):
        """
        Forward pass with CRF layer.
        
        Args:
            attention_mask: Boolean tensor where True = valid, False = padding
                        (already inverted in the main model)
        """
        if labels is not None:
            # Training mode: compute CRF loss
            loss = -self.crf(logits, labels, mask=attention_mask, reduction='mean')
            return {'loss': loss, 'logits': logits}
        else:
            # Inference mode: decode best sequence
            predictions = self.crf.decode(logits, mask=attention_mask)
            return {'predictions': predictions, 'logits': logits}

In [38]:
class CDSPredictor(nn.Module):
    """
    Model architecture for CDS prediction using a pretrained ESM-2 model and a Transformer head with CRF.
    """
    def __init__(self,
                 num_layers,
                 n_attention_heads,
                 d_model,
                 dropout_rate_1,
                 dropout_rate_2,
                 dropout_rate_3,
                 act_function,
                 num_encoded_labels,
                 encoded_labels_mapping,
                 crf_init_weight): 
        super(CDSPredictor, self).__init__()

        self.input_proj = nn.Linear(12, d_model)

        self.dropout_rate_1 = nn.Dropout(dropout_rate_1)

        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len=500)
        
        self.TransformerEncoderBlock = TransformerEncoderBlock(
            hidden_size=d_model,
            num_layers=num_layers,
            n_attention_heads=n_attention_heads,
            dropout_rate_encoder=dropout_rate_2,
            dropout_rate_final=dropout_rate_3,
            act_function=act_function,
            num_labels=6)
        
        ##Linear layer to go from 3*C -> num_encoded_labels
        self.output_proj = nn.Linear(3*6, num_encoded_labels)

        self.CRF = LinearChainCRF(mapping_dict_to_class = encoded_labels_mapping,
                                  crf_init_weight=crf_init_weight,
                                  num_class_labels=num_encoded_labels)

        self.relu = nn.ReLU()
        
    
    def forward(self, encoded_seqs_nt_rf0, 
                      encoded_seqs_nt_rf1,
                      encoded_seqs_nt_rf2,
                      labels=None):
        """
        Forward pass through the model with CRF support.
        
        Args:
            encoded_seqs_nt_rf0 (torch.Tensor): Encoded nucleotide sequences for RF0 [batch_size, seq_len, 12]
            attention_mask_rf0 (torch.Tensor): Attention mask for RF0 [batch_size, seq_len]
            encoded_seqs_nt_rf1 (torch.Tensor): Encoded nucleotide sequences for RF1 [batch_size, seq_len, 12]
            attention_mask_rf1 (torch.Tensor): Attention mask for RF1 [batch_size, seq_len]
            encoded_seqs_nt_rf2 (torch.Tensor): Encoded nucleotide sequences for RF2 [batch_size, seq_len, 12]
            attention_mask_rf2 (torch.Tensor): Attention mask for RF2 [batch_size, seq_len]
            labels (torch.Tensor, optional): True labels for CRF training [batch_size, seq_len]
            
        """

        padding_mask = create_codon_padding_mask(encoded_seqs_nt_rf0)

        encoded_embeddings_rf0 = self.dropout_rate_1(self.input_proj(encoded_seqs_nt_rf0))
        encoded_embeddings_rf1 = self.dropout_rate_1(self.input_proj(encoded_seqs_nt_rf1))
        encoded_embeddings_rf2 = self.dropout_rate_1(self.input_proj(encoded_seqs_nt_rf2))

        encoded_embeddings_rf0 = encoded_embeddings_rf0 + self.pos_encoding(encoded_embeddings_rf0)
        encoded_embeddings_rf1 = encoded_embeddings_rf1 + self.pos_encoding(encoded_embeddings_rf1)
        encoded_embeddings_rf2 = encoded_embeddings_rf2 + self.pos_encoding(encoded_embeddings_rf2)

        logits_rf0 = self.TransformerEncoderBlock(
            encoded_seqs_nt=encoded_embeddings_rf0,
            attention_mask=padding_mask)
        
        logits_rf1 = self.TransformerEncoderBlock(
            encoded_seqs_nt=encoded_embeddings_rf1,
            attention_mask=padding_mask)
        
        logits_rf2 = self.TransformerEncoderBlock(
            encoded_seqs_nt=encoded_embeddings_rf2,
            attention_mask=padding_mask) #output: [100, C]
        
        codon_embeddings = torch.cat([logits_rf0, logits_rf1, logits_rf2], dim=-1) #output: [100, 3*C]

        logits_encoded_labels = self.output_proj(codon_embeddings) #output: [100, num_encoded_labels]

        crf_mask = ~padding_mask  # Invert: True = valid, False = padding

        output = self.CRF(
            logits=logits_encoded_labels,
            attention_mask=crf_mask, #Input any trimmed attention mask; applies to same positions as before
            labels=labels)
        
        return output

In [39]:

def initialize_model(device, num_layers, n_attention_heads, d_model, dropout_rate_1, dropout_rate_2, dropout_rate_3, act_function, crf_init_weight):
    """
    Initialize the model and move it to the specified device.
    Args:
        device_type (str): The device to use for computation ("cuda", "mps", or "cpu").
    Returns:
        device (torch.device): The device being used.
        model (nn.Module): The initialized model.
    """
    
    print("Running on: ", device, flush = True)

    with open(f'{input_data_dir_path}/label_mappings/mapping_to_3d_vector.pkl', "rb") as mapping_file:
        mapping_dict_to_class = pickle.load(mapping_file)
    
    num_encoded_labels = len(mapping_dict_to_class.keys())
    print(f"Number of encoded label classes: {num_encoded_labels}")
    
    model = CDSPredictor(num_layers = num_layers,
                         n_attention_heads = n_attention_heads,
                         d_model=d_model,
                         dropout_rate_1 = dropout_rate_1, 
                         dropout_rate_2 = dropout_rate_2,
                         dropout_rate_3 = dropout_rate_3,
                         act_function = act_function,
                         num_encoded_labels = num_encoded_labels,
                         encoded_labels_mapping = mapping_dict_to_class,
                         crf_init_weight = crf_init_weight)
    model.to(device)
    
    if device.type == "cuda":
        print(f"Memory Allocated after loading model: {torch.cuda.memory_allocated(device) / 1024**3} GB")

    return model, mapping_dict_to_class


def print_model_dimensions(model):
    for name, param in model.named_parameters():
        print(f"{name}: {param.shape}")

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [40]:
def calculate_sequence_accuracy_metrics(true_labels_list, predictions_list, sequence_types_list=None):
    """
    Calculate sequence-level accuracy metrics including sensitivity, specificity, and MCC for overall and for each label specifically.
    Also calculates sequence type-specific metrics if sequence_types_list is provided.
    
    Args:
        true_labels_list: List of numpy arrays, each containing true labels for one sequence
        predictions_list: List of numpy arrays, each containing predictions for one sequence  
        sequence_types_list: Optional list of sequence types/descriptions corresponding to each sequence
        
    Returns:
        results (dict): Dictionary containing accuracy metrics including overall MCC and type-specific metrics
    """
    if len(true_labels_list) != len(predictions_list):
        raise ValueError("Mismatch in number of sequences")
    
    if sequence_types_list is not None and len(sequence_types_list) != len(true_labels_list):
        raise ValueError("Mismatch in number of sequence types")
    
    classes = [0, 1, 2, 3, 4, 5]
    total_sequences = len(true_labels_list)
    
    # Use more memory-efficient approach
    perfect_sequences = 0
    high_accuracy_sequences = 0
    sequence_accuracies = []
    
    # For sequence type-specific metrics
    type_specific_data = {}
    if sequence_types_list is not None:
        unique_types = list(set(sequence_types_list))
        for seq_type in unique_types:
            type_specific_data[seq_type] = {
                'true_labels': [],
                'predictions': [],
                'sequence_accuracies': [],
                'perfect_sequences': 0,
                'high_accuracy_sequences': 0,
                'total_sequences': 0
            }
    
    # Process sequences in batches to save memory
    batch_size = 1000  # Process 1000 sequences at a time
    all_true_labels = []
    all_predictions = []
    
    for i in range(0, total_sequences, batch_size):
        batch_true = true_labels_list[i:i+batch_size]
        batch_pred = predictions_list[i:i+batch_size]
        batch_types = sequence_types_list[i:i+batch_size] if sequence_types_list else None
        
        batch_accuracies = []
        batch_all_true = []
        batch_all_pred = []
        
        for idx, (true_labels, predictions) in enumerate(zip(batch_true, batch_pred)):
            true_labels = np.asarray(true_labels)
            predictions = np.asarray(predictions)
            
            # Add to overall arrays
            batch_all_true.extend(true_labels.tolist())
            batch_all_pred.extend(predictions.tolist())
            
            # Calculate accuracy
            accuracy = (true_labels == predictions).mean()
            batch_accuracies.append(accuracy)
            
            if accuracy == 1.0:
                perfect_sequences += 1
            if accuracy > 0.9:
                high_accuracy_sequences += 1
            
            # Handle type-specific data
            if batch_types is not None:
                seq_type = batch_types[idx]
                type_data = type_specific_data[seq_type]
                
                # Add to type-specific arrays
                type_data['true_labels'].extend(true_labels.tolist())
                type_data['predictions'].extend(predictions.tolist())
                type_data['sequence_accuracies'].append(accuracy)
                type_data['total_sequences'] += 1
                
                if accuracy == 1.0:
                    type_data['perfect_sequences'] += 1
                if accuracy > 0.9:
                    type_data['high_accuracy_sequences'] += 1
        
        sequence_accuracies.extend(batch_accuracies)
        all_true_labels.extend(batch_all_true)
        all_predictions.extend(batch_all_pred)
        
        # Clear batch data
        del batch_true, batch_pred, batch_accuracies, batch_all_true, batch_all_pred
        if batch_types is not None:
            del batch_types
    
    # Convert to numpy arrays
    all_true_labels = np.array(all_true_labels)
    all_predictions = np.array(all_predictions)
    
    # Calculate overall MCC efficiently
    try:
        overall_mcc = matthews_corrcoef(all_true_labels, all_predictions)
        if np.isnan(overall_mcc):
            overall_mcc = 0.0
    except:
        overall_mcc = 0.0
    
    # Calculate per-class accuracies more efficiently
    class_accuracies = {}
    for cls in classes:
        mask = all_true_labels == cls
        if mask.sum() > 0:
            class_accuracies[f'acc_class_{cls}'] = (all_predictions[mask] == all_true_labels[mask]).mean()
        else:
            class_accuracies[f'acc_class_{cls}'] = 0.0
    
    # Build results
    results = {
        'fraction_perfect_sequences': perfect_sequences / total_sequences,
        'fraction_high_accuracy_sequences': high_accuracy_sequences / total_sequences,
        'overall_mcc': overall_mcc,
        'accuracy': np.mean(sequence_accuracies),
        **class_accuracies
    }
    
    # Calculate type-specific metrics
    if sequence_types_list is not None:
        type_metrics = {}
        for seq_type, type_data in type_specific_data.items():
            if type_data['total_sequences'] > 0:
                # Convert type-specific data to numpy arrays
                type_true = np.array(type_data['true_labels'])
                type_pred = np.array(type_data['predictions'])
                
                # Calculate type-specific MCC
                try:
                    type_mcc = matthews_corrcoef(type_true, type_pred)
                    if np.isnan(type_mcc):
                        type_mcc = 0.0
                except:
                    type_mcc = 0.0
                
                # Calculate type-specific metrics
                type_metrics[f'{seq_type}_mcc'] = type_mcc
                type_metrics[f'{seq_type}_accuracy'] = np.mean(type_data['sequence_accuracies'])
                type_metrics[f'{seq_type}_fraction_perfect'] = type_data['perfect_sequences'] / type_data['total_sequences']
                type_metrics[f'{seq_type}_fraction_high_accuracy'] = type_data['high_accuracy_sequences'] / type_data['total_sequences']
                
                # Calculate per-class accuracies for this type
                for cls in classes:
                    mask = type_true == cls
                    if mask.sum() > 0:
                        type_metrics[f'{seq_type}_acc_class_{cls}'] = (type_pred[mask] == type_true[mask]).mean()
                    else:
                        type_metrics[f'{seq_type}_acc_class_{cls}'] = 0.0
        
        # Add type-specific metrics to results
        results.update(type_metrics)
    
    return results

## Track loss of different sequence types, adjust distribution of training samples accordingly

In [41]:
class CategoricalLossTracker:
    """ 
    A class to track losses for different categories during training.
    """
    def __init__(self, categories):
        self.categories = categories
        self.losses = {cat: [] for cat in categories} # Create empty list for each category
            
    def update(self, category, loss):
        #Extract average loss for the category in a batch
        self.losses[category].append(loss.item())
            
    def get_metrics(self):
        # Calculate the mean loss for each category
        return {cat: np.mean(losses) for cat, losses in self.losses.items()}
    

def create_weighted_sampler(dataset, type_weights):
    """
    Create a weighted sampler based on sequence types and their weights.
    """
    # Get sequence types for all samples in dataset
    seq_types = [dataset[i]['seq_desc'] for i in range(len(dataset))]
    
    # Create sample weights based on type_weights
    sample_weights = [type_weights.get(seq_type, 1.0) for seq_type in seq_types]
    
    # Create WeightedRandomSampler
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(dataset),
        replacement=True
    )
    
    return sampler

In [42]:
def adjust_train_sample_distribution(train_data, train_sampler, batch_size):
    train_loader = DataLoader(
        train_data, 
        batch_size=batch_size, 
        sampler=train_sampler,
        num_workers=num_workers_cpu, 
        pin_memory=pin_memory,
        drop_last=True
    )

    return train_loader

In [43]:
def log_evaluation_metrics(epoch, train_avg_loss, val_avg_loss, best_val_loss, tracker, sequence_metrics, val_times_counter, sequence_types, sequence_type_weights):
    # Build the print statement with type-specific metrics
    print_parts = [
        f"---Evaluation {val_times_counter}---\n",
        f"Train Loss: {train_avg_loss:.4f}\t\t",
        f"Val Loss: {val_avg_loss:.4f}\t\t"
    ]
    
    # Add overall metrics
    print_parts.extend([
        f"Overall MCC: {sequence_metrics.get('overall_mcc', 0):.4f}\t\t",
        f"Overall Accuracy: {sequence_metrics.get('accuracy', 0):.4f}\n"
    ])
    
    # Add loss metrics for each sequence type
    for seq_type in sequence_types:
        loss_val = tracker.get_metrics().get(seq_type, 0)
        print_parts.append(f"Val Loss {seq_type}: {loss_val:.4f}\t\t")
    
    print_parts.append("\n")
    
    # Add type-specific MCC and accuracy metrics
    for seq_type in sequence_types:
        type_mcc = sequence_metrics.get(f'{seq_type}_mcc', 0)
        type_acc = sequence_metrics.get(f'{seq_type}_accuracy', 0)
        if type_mcc != 0 or type_acc != 0:  # Only show if we have data for this type
            print_parts.extend([
                f"MCC {seq_type}: {type_mcc:.4f}\t\t",
                f"Acc {seq_type}: {type_acc:.4f}\t\t"
            ])
    
    # Print all metrics
    print("".join(print_parts), flush=True)
    
    # Build wandb logging dictionary
    wandb_log = {
        "epoch": epoch + 1,
        "train_loss": train_avg_loss,
        "val_loss": val_avg_loss,
        
        # Overall sequence metrics
        "val_fraction_perfect_sequences": sequence_metrics.get('fraction_perfect_sequences', 0),
        "val_fraction_high_accuracy_sequences": sequence_metrics.get('fraction_high_accuracy_sequences', 0),
        "val_overall_mcc": sequence_metrics.get('overall_mcc', 0),
        "val_accuracy": sequence_metrics.get('accuracy', 0)}
    
    val_loss_weighted_average = 0

    # Add loss metrics for each sequence type
    for seq_type in sequence_types:
        wandb_log[f"val_loss_{seq_type}"] = tracker.get_metrics().get(seq_type, 0)
        val_loss_weighted_average += tracker.get_metrics().get(seq_type, 0) * sequence_type_weights[seq_type] #DELETE 
    
    # Add type-specific MCC and accuracy metrics
    for seq_type in sequence_types:
        # MCC metrics
        type_mcc = sequence_metrics.get(f'{seq_type}_mcc', 0)
        if type_mcc != 0:  # Only log if we have data
            wandb_log[f"val_mcc_{seq_type}"] = type_mcc

        # Accuracy metrics
        type_acc = sequence_metrics.get(f'{seq_type}_accuracy', 0)
        if type_acc != 0:  # Only log if we have data
            wandb_log[f"val_accuracy_{seq_type}"] = type_acc
        
        # Perfect and high accuracy sequence fractions
        type_perfect = sequence_metrics.get(f'{seq_type}_fraction_perfect', 0)
        if type_perfect != 0:
            wandb_log[f"val_fraction_perfect_{seq_type}"] = type_perfect
            
        type_high_acc = sequence_metrics.get(f'{seq_type}_fraction_high_accuracy', 0)
        if type_high_acc != 0:
            wandb_log[f"val_fraction_high_accuracy_{seq_type}"] = type_high_acc
        
    wandb_log[f"val_loss_weighted_average"] = val_loss_weighted_average #DELETE
    wandb_log[f"best_val_loss"] = best_val_loss

    # Log to wandb
    wandb.log(wandb_log)
    
    return val_times_counter + 1

In [44]:
def training_iteration(i, batch, scaler, model, optimizer, device, epoch_losses):
    # Move data to device
    inputs_nt_rf0 = batch["nt_encodings_rf0"].to(device, non_blocking=True)
    inputs_nt_rf1 = batch["nt_encodings_rf1"].to(device, non_blocking=True)
    inputs_nt_rf2 = batch["nt_encodings_rf2"].to(device, non_blocking=True)
    
    encoded_labels = batch['label_encodings'].to(device, non_blocking=True).long()

    # Set all padding positions to a valid label (e.g., 0)
    encoded_labels = encoded_labels.clone()
    encoded_labels[encoded_labels == -1] = 0 #-1 is padding label

    # Forward pass - use mixed precision if available
    if scaler is not None:
        with autocast("cuda"):
            outputs = model(inputs_nt_rf0,
                            inputs_nt_rf1,
                            inputs_nt_rf2,
                            labels=encoded_labels)
            loss = outputs['loss']
    else:
        # Regular forward pass without mixed precision
        outputs = model(inputs_nt_rf0,
                        inputs_nt_rf1,
                        inputs_nt_rf2,
                        labels=encoded_labels)
        loss = outputs['loss']

    # Backward pass - handle both mixed precision and regular training
    optimizer.zero_grad()
        
    if scaler is not None:
        # Mixed precision backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        # Regular backward pass
        loss.backward()
        optimizer.step()

    # Store loss value and cleanup immediately
    epoch_losses.append(loss.item())

    #Periodic cleanup
    #if i % 20 == 0:
    #    clear_memory()

    if i % 100000 == 0:
        print(model.CRF.crf.transitions)
        
    # Immediate cleanup to prevent memory leaks
    del outputs, loss
    del inputs_nt_rf0, inputs_nt_rf1, inputs_nt_rf2, encoded_labels

    return epoch_losses

In [45]:
def show_examples(counter, v_labels, padding_mask, predictions, seq_descs_batch, mapping_dict_to_class):
    # Show hard examples as demonstration
    if counter == 0:
        for seq_i in range(min(32, v_labels.shape[0])):
            mask = ~padding_mask[seq_i]
            if seq_descs_batch[seq_i] not in ["non-coding", "coding", "coding_with_substitutions"]:
                print("Sequence type:", seq_descs_batch[seq_i])
                
                # Convert labels to RF vectors
                labels_masked = v_labels[seq_i][mask].cpu().numpy().astype(int)
                labels_rf = [mapping_dict_to_class[label] for label in labels_masked]
                
                # Convert predictions to RF vectors
                preds_masked = predictions[seq_i][mask].cpu().numpy().astype(int)
                preds_rf = [mapping_dict_to_class[pred] for pred in preds_masked]
                
                # Extract individual RF sequences
                labels_rf0 = [rf[0] for rf in labels_rf]
                labels_rf1 = [rf[1] for rf in labels_rf]
                labels_rf2 = [rf[2] for rf in labels_rf]
                
                preds_rf0 = [rf[0] for rf in preds_rf]
                preds_rf1 = [rf[1] for rf in preds_rf]
                preds_rf2 = [rf[2] for rf in preds_rf]

                print("Encoded labels and preds:")
                print(v_labels[seq_i][mask].cpu().numpy().astype(float))
                print(predictions[seq_i][mask].cpu().numpy().astype(float))
                print()
                
                print("Labels RF0:", labels_rf0)
                print("Labels RF1:", labels_rf1)
                print("Labels RF2:", labels_rf2)

                print()
                print("Predictions RF0:", preds_rf0)
                print("Predictions RF1:", preds_rf1)
                print("Predictions RF2:", preds_rf2)
                print("\n")

In [ ]:
def show_examples(counter, v_labels, padding_mask, logits, seq_descs_batch, mapping_dict_to_class, model, device, valid_mask):
    """
    Show hard examples as demonstration - only decode sequences we want to display
    
    Args:
        counter: batch counter
        v_labels: encoded labels
        padding_mask: padding mask
        logits: model logits (not predictions)
        seq_descs_batch: sequence descriptions
        mapping_dict_to_class: label mapping
        model: model object (to access CRF)
        device: device
        valid_mask: valid positions mask
    """
    if counter == 0:
        num_to_show = min(132, v_labels.shape[0])  # Show examples
            
        for seq_i in range(num_to_show):
            mask = ~padding_mask[seq_i]
                
            if seq_descs_batch[seq_i] not in ["non-coding", "coding", "coding_with_substitutions"]:
                print("Sequence type:", seq_descs_batch[seq_i])
                
                # Get labels for this sequence
                labels_masked = v_labels[seq_i][mask].cpu().numpy().astype(int)
                    
                # Decode prediction for ONLY this sequence
                seq_logits = logits[seq_i:seq_i+1]  # [1, seq_len, num_classes]
                seq_valid_mask = valid_mask[seq_i:seq_i+1]  # [1, seq_len]
                    
                # Decode only this one sequence
                pred_decoded = model.CRF.crf.decode(seq_logits, mask=seq_valid_mask)
                preds_masked = torch.tensor(pred_decoded[0], dtype=torch.long, device=device).cpu().numpy().astype(int)
                    
                # Convert labels to RF vectors
                labels_rf = [mapping_dict_to_class[label] for label in labels_masked]
                    
                # Convert predictions to RF vectors
                preds_rf = [mapping_dict_to_class[pred] for pred in preds_masked]
                    
                # Extract individual RF sequences
                labels_rf0 = [rf[0] for rf in labels_rf]
                labels_rf1 = [rf[1] for rf in labels_rf]
                labels_rf2 = [rf[2] for rf in labels_rf]
                preds_rf0 = [rf[0] for rf in preds_rf]
                preds_rf1 = [rf[1] for rf in preds_rf]
                preds_rf2 = [rf[2] for rf in preds_rf]
                    
                print("Encoded labels and preds:")
                print(v_labels[seq_i][mask].cpu().numpy().astype(float))
                print(preds_masked.astype(float))
                print()
                print("Labels RF0:", labels_rf0)
                print("Labels RF1:", labels_rf1)
                print("Labels RF2:", labels_rf2)
                print()
                print("Predictions RF0:", preds_rf0)
                print("Predictions RF1:", preds_rf1)
                print("Predictions RF2:", preds_rf2)
                print("\n")


In [47]:
def objective(trial, train_data, val_data, sequence_types, seq_type_desc_fracs):
    #Define hyperparameter ranges to sample from
    depths_transformer_encoder_blocks = [2, 4, 6, 8, 10, 12]
    attention_heads = [2, 4, 8]
    d_model_sizes = [64, 128, 256, 512, 1024]
    dropout_rates_1 = [0.1, 0.2, 0.3] #Dropout rate applied after input projection
    dropout_rates_2 = [0.2, 0.3, 0.4, 0.5] #dropout rate applied in transformer encoder layers
    dropout_rates_3 = [0.0, 0.05] 
    lr_scratch_rates = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5] 
    act_functions = ["relu", "gelu"]
    weight_decays = [0.001, 0.01, 0.1]
    crf_init_weights = [1, 10]

    #Define trial suggestions; set hyperparameters
    depth_transformer_encoder_blocks = trial.suggest_categorical('depth_transformer_encoder_blocks', depths_transformer_encoder_blocks)
    n_attention_heads = trial.suggest_categorical('n_attention_heads', attention_heads)
    d_model = trial.suggest_categorical('d_model', d_model_sizes)
    dropout_rate_1 = trial.suggest_categorical('dropout_rate_1', dropout_rates_1)
    dropout_rate_2 = trial.suggest_categorical('dropout_rate_2', dropout_rates_2)
    dropout_rate_3 = trial.suggest_categorical('dropout_rate_3', dropout_rates_3)
    lr_scratch = trial.suggest_categorical('lr_scratch', lr_scratch_rates)
    act_function = trial.suggest_categorical('act_function', act_functions)
    weight_decay = trial.suggest_categorical('weight_decay', weight_decays)
    crf_init_weight = trial.suggest_categorical('crf_init_weight', crf_init_weights)

    batch_size = 64

    print(f"depth_transformer_encoder_blocks: {depth_transformer_encoder_blocks}\n \
            n_attention_heads: {n_attention_heads}\n \
            dropout_rate_1: {dropout_rate_1}\n \
            dropout_rate_2: {dropout_rate_2}\n \
            dropout_rate_3: {dropout_rate_3}\n \
            lr_scratch: {lr_scratch}\n \
            act_function: {act_function}\n \
            weight_decay: {weight_decay}\n \
            crf_init_weight: {crf_init_weight}")

    wandb.init(project=wandb_project_name, 
                config={
                        "depth_transformer_encoder_blocks": depth_transformer_encoder_blocks,
                        "n_attention_heads": n_attention_heads,
                        "d_model": d_model,
                        "dropout_rate_1": dropout_rate_1,
                        "dropout_rate_2": dropout_rate_2,
                        "dropout_rate_3": dropout_rate_3,
                        "act_function": act_function,
                        "weight_decay": weight_decay,
                        "crf_init_weight": crf_init_weight,
                        "lr_scratch": lr_scratch},
                        name = f"trial_{trial.number}")
    
    type_weights = {st: 1.0 for st in sequence_types}  # Initial sampling weights
    min_weight = 0.5  # Minimum weight to keep (avoid complete removal)

    min_weight_sum = min_weight * len(sequence_types) # Minimum sum of weights to keep
    print(f"Minimum sum of weights before stopping: {min_weight_sum}")

    # Create initial weighted sampler
    train_sampler = create_weighted_sampler(train_data, type_weights)

    #Define data loaders
    train_loader = adjust_train_sample_distribution(train_data, train_sampler, batch_size)

    val_loader = DataLoader(
        val_data, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers_cpu,  
        pin_memory=pin_memory)

    model, mapping_dict_to_class = initialize_model(device,
                            num_layers = depth_transformer_encoder_blocks,
                            n_attention_heads = n_attention_heads,
                            d_model=d_model,
                            dropout_rate_1=dropout_rate_1,
                            dropout_rate_2=dropout_rate_2,
                            dropout_rate_3=dropout_rate_3,
                            act_function = act_function,
                            crf_init_weight = crf_init_weight)
        
    total_params = count_parameters(model)
    print(f"Total trainable parameters: {total_params:,}")
    #print(f"Model dimensions: {print_model_dimensions(model)}")

    #Define settings for training
    epochs = 30
    steps_per_epoch = len(train_data) / batch_size                               #The number of steps taken per epoch
    eval_per_epoch = 6 #MODIFY                                                          #The number of evaluations made per epoch
    steps_between_evals = int(steps_per_epoch / eval_per_epoch)                  #The number of steps to take between each evaluation

    #Initialize the loss tracker
    tracker = CategoricalLossTracker(sequence_types)

    #Define the optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr_scratch, weight_decay=weight_decay)

    #Initialize variables for early stopping
    best_val_loss = float('inf')
    threshold_patience = 6  #Number of evaluations with no improvement to wait before stopping #MODIFY
    counter_patience = 0

    #Initialize variables for training loop
    step = 0  # Global step counter
    val_times_counter = 0

    # Initialize mixed precision scaler if using CUDA
    scaler = GradScaler() if "cuda" in device_type else None
    if scaler is not None:
        print("Mixed precision training enabled")

    #Training loop
    for epoch in range(epochs):
        model.train()
        epoch_losses = []  # Reset at start of each epoch
        batches_processed = 0  # Track actual batches processed this epoch
        
        for i, batch in enumerate(train_loader):
            #Run training iteration
            epoch_losses = training_iteration(i, batch, scaler, model, optimizer, device, epoch_losses)
            batches_processed += 1
            
            #Validation step
            if step % steps_between_evals == 0 and step > 0:
                clear_memory()
                model.eval()
                val_losses = []
                all_val_true_sequences = []
                all_val_pred_sequences = []
                all_val_sequence_types = [] 
                
                with torch.no_grad():
                    for counter, val_batch in enumerate(val_loader):
                        v_inputs_nt_rf0 = val_batch["nt_encodings_rf0"].to(device, non_blocking=True)
                        v_inputs_nt_rf1 = val_batch["nt_encodings_rf1"].to(device, non_blocking=True)
                        v_inputs_nt_rf2 = val_batch["nt_encodings_rf2"].to(device, non_blocking=True)
                            
                        v_encoded_labels = val_batch['label_encodings'].to(device, non_blocking=True).long()
                            
                        # Save padding mask before overwriting v_labels
                        padding_mask = (v_encoded_labels == -1)
                        valid_mask = ~padding_mask  # This is what we'll use consistently

                        # Set all padding positions to a valid label (e.g., 0)
                        # Create modified labels for model forward pass
                        v_labels_modified = v_encoded_labels.clone()
                        v_labels_modified[padding_mask] = 0  # Set padding to valid label for forward pass


                        # Forward pass for validation
                        if scaler is not None:
                            with autocast("cuda"):
                                v_outputs = model(v_inputs_nt_rf0, 
                                                v_inputs_nt_rf1,
                                                v_inputs_nt_rf2, 
                                                v_labels_modified)
                                v_loss = v_outputs['loss']
                        else:
                            v_outputs = model(v_inputs_nt_rf0, 
                                            v_inputs_nt_rf1, 
                                            v_inputs_nt_rf2, 
                                            v_labels_modified)
                            v_loss = v_outputs['loss']
                        
                        val_losses.append(v_loss.item())
                        
                        logits_for_metrics = v_outputs['logits']  # Save logits instead of decoding all predictions

                        # Process category-specific losses (keep your existing logic)
                        seq_descs_batch = val_batch['seq_desc']

                        for desc in tracker.categories:
                            # Find which sequences in the batch belong to this desc
                            desc_mask = torch.tensor([d == desc for d in seq_descs_batch], device=device)
                            if desc_mask.any():
                                # Select only the relevant sequences
                                desc_logits = v_outputs["logits"][desc_mask]           # [num_desc, seq_len, 4]
                                
                                # IMPORTANT: Use the same label preprocessing as the main loss
                                desc_labels_original = v_encoded_labels[desc_mask]     # Original labels
                                desc_labels_modified = v_labels_modified[desc_mask]     # Modified labels (padding=0)
                                desc_valid_mask = valid_mask[desc_mask]                 # Valid positions mask
                                
                                # Option 1: Use modified labels (consistent with forward pass)
                                desc_crf_loss = -model.CRF.crf(desc_logits, desc_labels_modified, 
                                                            mask=desc_valid_mask, reduction='mean')
                                
                                tracker.update(desc, desc_crf_loss)

                                del desc_logits, desc_labels_original, desc_labels_modified, desc_valid_mask, desc_crf_loss

                            del desc_mask
                        
                        for seq_idx in range(v_encoded_labels.shape[0]):
                            mask = ~padding_mask[seq_idx]  # Use NOT padding_mask to select valid positions
                            if mask.any():  # Skip empty sequences
                                true_seq = v_encoded_labels[seq_idx][mask].cpu().numpy()
                                
                                # Only decode prediction for this sequence
                                seq_logits = logits_for_metrics[seq_idx:seq_idx+1]  # [1, seq_len, num_classes]
                                seq_valid_mask = valid_mask[seq_idx:seq_idx+1]
                                pred_decoded = model.CRF.crf.decode(seq_logits, mask=seq_valid_mask)
                                pred_seq = np.array(pred_decoded[0])
                                
                                seq_type = seq_descs_batch[seq_idx]

                                all_val_true_sequences.append(true_seq)
                                all_val_pred_sequences.append(pred_seq)
                                all_val_sequence_types.append(seq_type)

                        #Monitor hard examples as demonstration of how well classification is going
                        show_examples(counter, v_encoded_labels, padding_mask, logits_for_metrics, seq_descs_batch, mapping_dict_to_class, model, device, valid_mask)
                        

                        # Clean up each validation batch immediately
                        del v_inputs_nt_rf0, v_inputs_nt_rf1, v_inputs_nt_rf2
                        del v_encoded_labels, v_labels_modified
                        del v_outputs, v_loss, padding_mask

                        #Aggressive cleanup every 10 batches
                        #if counter % 10 == 0:
                        #    clear_memory()


                # Calculate validation loss and other final metrics for validation loop
                if val_losses:
                    val_avg_loss = sum(val_losses) / len(val_losses)
                else:
                    val_avg_loss = float('inf')

                #Implement pruning strategy
                trial.report(val_avg_loss, step)

                #Handle pruning based on the intermediate value.
                if trial.should_prune():
                    raise optuna.TrialPruned()
                
                if all_val_true_sequences and all_val_pred_sequences:
                    sequence_metrics = calculate_sequence_accuracy_metrics(all_val_true_sequences, 
                                                                        all_val_pred_sequences,
                                                                        all_val_sequence_types)
                else:
                    sequence_metrics = {}
                
                # Calculate training loss - use recent batches only
                if epoch_losses:
                    train_avg_loss = sum(epoch_losses[-steps_between_evals:]) / min(len(epoch_losses), steps_between_evals)
                else:
                    train_avg_loss = 0.0
                
                # Early stopping check
                if val_avg_loss < best_val_loss:
                    best_val_loss = val_avg_loss
                    #torch.save(model.state_dict(), f"../../../data/processed_data/model_data/single_rf_crf/model_without_errors/models/{project_name}_trained.pth")
                    counter_patience = 0
                else:
                    counter_patience += 1
                    
                if counter_patience >= threshold_patience:
                    print("Early stopping triggered!")
                    break

                #Log metrics
                val_times_counter = log_evaluation_metrics(epoch, train_avg_loss, val_avg_loss, best_val_loss, tracker, sequence_metrics, val_times_counter, sequence_types, seq_type_desc_fracs)

                #Clean up validation data
                del val_losses, all_val_true_sequences, all_val_pred_sequences
                clear_memory()
                
                model.train()  # Back to training mode
            
            # Increment global step counter after each batch
            step += 1
        
        # Check if early stopping was triggered during validation
        if counter_patience >= threshold_patience:
            break

    wandb.finish()
    return best_val_loss

In [ ]:
np.random.seed(42)

train_data, val_data, sequence_types, seq_type_desc_fracs = load_and_process_data(max_len)

train_indices = np.random.choice(len(train_data), size=min(32*1000, len(train_data)), replace=False)
val_indices = np.random.choice(len(val_data), size=min(32*100, len(val_data)), replace=False)

train_data = torch.utils.data.Subset(train_data, train_indices.tolist())
val_data = torch.utils.data.Subset(val_data, val_indices.tolist())

Training data samples:  41756
Validation data samples:  46050


/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_29115/1399213979.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(encoding)


In [49]:

#Create a study object and optimize the objective function
study = optuna.create_study(direction='minimize',   #Minimize loss
                            pruner=optuna.pruners.HyperbandPruner())
study.optimize(
    lambda trial: objective(trial, train_data, val_data, sequence_types, seq_type_desc_fracs), 
    n_trials=30
)

model_config = {
    "depth_transformer_encoder_blocks": study.best_params['depth_transformer_encoder_blocks'],
    "n_attention_heads": study.best_params['n_attention_heads'],
    "d_model": study.best_params['d_model'],
    "dropout_rate_1": study.best_params['dropout_rate_1'],
    "dropout_rate_2": study.best_params['dropout_rate_2'],
    "dropout_rate_3": study.best_params['dropout_rate_3'],
    "lr_scratch": study.best_params['lr_scratch'],
    "act_function": study.best_params['act_function'],
    "weight_decay": study.best_params['weight_decay'],
    "crf_init_weight": study.best_params['crf_init_weight']}

os.makedirs("../../../data/processed_data/model_data/shared_crf/model_with_errors/hyperparameter_configs/", exist_ok = True)

#Save the dictionary as a JSON file
file_path = f'../../../data/processed_data/model_data/shared_crf/model_with_errors/hyperparameter_configs/nt_encoding_hyperparameters.json'
with open(file_path, 'w') as json_file:
    json.dump(model_config, json_file, indent=4)

print(f"Model configuration saved to {file_path}.")

[I 2025-10-22 09:51:51,722] A new study created in memory with name: no-name-a71edcf0-f9fe-4faa-b867-200c50ef77fd
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


depth_transformer_encoder_blocks: 2
             n_attention_heads: 2
             dropout_rate_1: 0.1
             dropout_rate_2: 0.3
             dropout_rate_3: 0.05
             lr_scratch: 0.001
             act_function: relu
             weight_decay: 0.1
             crf_init_weight: 1


wandb: Currently logged in as: linesandvad (linesandvad-university-of-copenhagen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Minimum sum of weights before stopping: 3.5
Running on:  mps
Number of encoded label classes: 79
Creating constraint mask for 79-label CRF...
Legal transitions: 826
Illegal transitions: 5415
Total transitions: 6241
Percentage legal: 13.2%
Total trainable parameters: 407,138


/Users/nrt204/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


...

Sequence type: transition_start
Encoded labels and preds:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0.]

Labels RF0: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Labels RF1: [0, 0, 0, 0, 

[W 2025-10-22 09:58:45,087] Trial 0 failed with parameters: {'depth_transformer_encoder_blocks': 2, 'n_attention_heads': 2, 'd_model': 128, 'dropout_rate_1': 0.1, 'dropout_rate_2': 0.3, 'dropout_rate_3': 0.05, 'lr_scratch': 0.001, 'act_function': 'relu', 'weight_decay': 0.1, 'crf_init_weight': 1} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/nrt204/Library/Python/3.9/lib/python/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_29115/242232322.py", line 5, in <lambda>
    lambda trial: objective(trial, train_data, val_data, sequence_types, seq_type_desc_fracs),
  File "/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_29115/2259209237.py", line 119, in objective
    epoch_losses = training_iteration(i, batch, scaler, model, optimizer, device, epoch_losses)
  File "/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h000

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x1095f5a60>> (for post_run_cell), with arguments args (<ExecutionResult object at 1095fe9a0, execution_count=49 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 1095fecd0, raw_cell="
#Create a study object and optimize the objective.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/nrt204/Desktop/FragmentPredictor/scripts/modeling/modeling_shared_crf/hyperparameter_tuning_with_errors_nt_encoding.ipynb#X35sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe